# 07D — 3-Agent Consensus + Error Analysis — FINAL V2

Input utama adalah `main_evaluator_final_477.csv` dari 07C FINAL V2.

Majority voting dilakukan terpisah pada:
- `error_type`
- `severity`
- `verdict`

Aturan:
- 3/3 sama → unanimous majority
- 2/3 sama → majority
- 1-1-1 → `__no_majority__`

Tidak ada label yang dipaksakan jika tidak ada majority.
Kasus `__no_majority__` atau tuple konsensus yang tidak konsisten secara logis diberi `needs_manual_review=True`.

Setelah voting selesai, identitas skenario baru digabungkan dari mapping privat. Reference test tetap tidak diperlukan pada tahap LLM evaluator.

## 1. Environment + latest COMPLETE 07C

In [1]:
from __future__ import annotations

import json
import os
from collections import Counter
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

PANEL_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "llm_evaluator_main"
    / "panel_runs"
)

PREP_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "llm_evaluator_main"
    / "prepared"
)

MAPPING_PATH = (
    PREP_DIR
    / "main_evaluator_private_mapping.csv"
)

BLIND_PATH = (
    PREP_DIR
    / "main_evaluator_blind_159.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "llm_evaluator_main"
    / "consensus"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

candidate_runs = sorted(
    PANEL_ROOT.glob(
        "run_*/main_evaluator_final_477.csv"
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

FINAL_477_PATH = None
judgments = None

for path in candidate_runs:
    try:
        df = pd.read_csv(
            path,
            dtype={
                "eval_case_id": str,
                "requested_model": str,
            },
            keep_default_na=False,
            encoding="utf-8-sig",
        )

        if (
            len(df) == 477
            and df["eval_case_id"].nunique() == 159
            and df["requested_model"].nunique() == 3
            and bool(
                (
                    df.groupby(
                        "requested_model"
                    ).size()
                    == 159
                ).all()
            )
        ):
            FINAL_477_PATH = path
            judgments = df
            break

    except Exception:
        continue

if FINAL_477_PATH is None:
    raise FileNotFoundError(
        "Tidak ditemukan main_evaluator_final_477.csv COMPLETE."
    )

mapping = pd.read_csv(
    MAPPING_PATH,
    dtype={
        "eval_case_id": str,
        "row_id": str,
    },
    keep_default_na=False,
    encoding="utf-8-sig",
)

blind = pd.read_csv(
    BLIND_PATH,
    dtype={
        "eval_case_id": str,
    },
    keep_default_na=False,
    encoding="utf-8-sig",
)

if len(mapping) != 159:
    raise RuntimeError(
        "Private mapping bukan 159."
    )

if len(blind) != 159:
    raise RuntimeError(
        "Blind input bukan 159."
    )

print("FINAL 477:", FINAL_477_PATH)
print("Judgments:", len(judgments))

FINAL 477: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/panel_runs/run_f23f6109bec1a27d/main_evaluator_final_477.csv
Judgments: 477


## 2. Majority consensus per atribut

In [2]:
ATTRIBUTES = [
    "error_type",
    "severity",
    "verdict",
]

PRED_COLUMNS = {
    "error_type":
        "pred_error_type",
    "severity":
        "pred_severity",
    "verdict":
        "pred_verdict",
}

def majority(values):
    vals = [
        str(v)
        for v in values
        if str(v).strip()
    ]

    if len(vals) != 3:
        raise RuntimeError(
            f"Expected 3 values, found {len(vals)}."
        )

    counts = Counter(vals)
    label, count = (
        counts.most_common(1)[0]
    )

    if count >= 2:
        return label, int(count)

    return "__no_majority__", 1

consensus_rows = []

for eval_case_id, group in judgments.groupby(
    "eval_case_id",
    sort=False,
):
    if len(group) != 3:
        raise RuntimeError(
            f"{eval_case_id} bukan 3 evaluator."
        )

    result = {
        "eval_case_id":
            eval_case_id,
    }

    for attr in ATTRIBUTES:
        col = PRED_COLUMNS[attr]
        value, votes = majority(
            group[col].tolist()
        )

        result[
            f"consensus_{attr}"
        ] = value

        result[
            f"{attr}_votes"
        ] = votes

        result[
            f"{attr}_all3_agree"
        ] = (
            group[col].nunique()
            == 1
        )

    ce = result[
        "consensus_error_type"
    ]
    cs = result[
        "consensus_severity"
    ]
    cv = result[
        "consensus_verdict"
    ]

    has_no_majority = (
        "__no_majority__"
        in {ce, cs, cv}
    )

    logical_valid = (
        not has_no_majority
    )

    if logical_valid:
        if ce == "correct":
            logical_valid = (
                cs == "none"
                and cv == "layak"
            )
        elif cs == "none":
            logical_valid = False
        elif (
            cs in {
                "major",
                "critical",
            }
            and cv
            != "perlu_ditinjau"
        ):
            logical_valid = False
        elif ce == "empty_output":
            logical_valid = (
                cs == "critical"
                and cv
                == "perlu_ditinjau"
            )

    result[
        "consensus_logical_valid"
    ] = bool(
        logical_valid
    )

    result[
        "needs_manual_review"
    ] = bool(
        has_no_majority
        or not logical_valid
    )

    group_sorted = group.sort_values(
        "panel_position"
    )

    exact_tuple = group_sorted[
        (
            group_sorted[
                "pred_error_type"
            ]
            == ce
        )
        &
        (
            group_sorted[
                "pred_severity"
            ]
            == cs
        )
        &
        (
            group_sorted[
                "pred_verdict"
            ]
            == cv
        )
    ]

    representative = (
        exact_tuple.iloc[0]
        if not exact_tuple.empty
        else group_sorted.iloc[0]
    )

    result[
        "representative_reason"
    ] = representative[
        "pred_reason"
    ]

    for row in group_sorted.itertuples(
        index=False
    ):
        pos = int(
            row.panel_position
        )
        result[
            f"agent{pos}_model"
        ] = row.display_name
        result[
            f"agent{pos}_error_type"
        ] = row.pred_error_type
        result[
            f"agent{pos}_severity"
        ] = row.pred_severity
        result[
            f"agent{pos}_verdict"
        ] = row.pred_verdict
        result[
            f"agent{pos}_reason"
        ] = row.pred_reason

    consensus_rows.append(
        result
    )

consensus = pd.DataFrame(
    consensus_rows
)

if len(consensus) != 159:
    raise RuntimeError(
        "Consensus harus 159."
    )

display(consensus.head())

,eval_case_id,consensus_error_type,error_type_votes,error_type_all3_agree,consensus_severity,severity_votes,severity_all3_agree,consensus_verdict,verdict_votes,verdict_all3_agree,...,agent2_model,agent2_error_type,agent2_severity,agent2_verdict,agent2_reason,agent3_model,agent3_error_type,agent3_severity,agent3_verdict,agent3_reason
0,EVAL001,meaning_shift,3,True,major,3,True,perlu_ditinjau,3,True,...,GPT-5.2,meaning_shift,major,perlu_ditinjau,"Kandidat menerjemahkan ""malillin"" sebagai ""mem...",Claude Sonnet 4.6,meaning_shift,major,perlu_ditinjau,Kata 'malillin' kemungkinan berarti 'memandika...
1,EVAL002,hallucination,2,False,major,2,False,perlu_ditinjau,3,True,...,GPT-5.2,hallucination,major,perlu_ditinjau,Kandidat menambahkan informasi seperti “banyak...,Claude Sonnet 4.6,meaning_shift,major,perlu_ditinjau,Frasa 'lako bokoqna' yang berarti 'ke arah bel...
2,EVAL003,untranslated,3,True,critical,2,False,perlu_ditinjau,3,True,...,GPT-5.2,untranslated,major,perlu_ditinjau,Kandidat masih sepenuhnya berisi kalimat Bahas...,Claude Sonnet 4.6,untranslated,critical,perlu_ditinjau,Kandidat terjemahan identik dengan source Baha...
3,EVAL004,__no_majority__,1,False,__no_majority__,1,False,layak,2,False,...,GPT-5.2,correct,none,layak,Kandidat mempertahankan makna utama sumber: me...,Claude Sonnet 4.6,meaning_shift,major,perlu_ditinjau,Frasa 'teqda kitaq kuissenni' lebih tepat dite...
4,EVAL005,hallucination,3,True,critical,3,True,perlu_ditinjau,3,True,...,GPT-5.2,hallucination,critical,perlu_ditinjau,Kalimat kandidat membahas tidak tahu dan menyi...,Claude Sonnet 4.6,hallucination,critical,perlu_ditinjau,Source 'Umbori joq too?' tampaknya merupakan k...


## 3. Merge mapping privat setelah voting

In [3]:
case_level = (
    consensus
    .merge(
        mapping,
        on="eval_case_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        blind[
            [
                "eval_case_id",
                "source_text",
                "candidate_text",
            ]
        ],
        on="eval_case_id",
        how="left",
        validate="one_to_one",
    )
)

if len(case_level) != 159:
    raise RuntimeError(
        "Case-level merge bukan 159."
    )

scenario_counts = (
    case_level[
        "scenario_slug"
    ].value_counts()
)

print(scenario_counts)
display(case_level.head())

scenario_slug
fsp_locked     53
lora_locked    53
zero_shot      53
Name: count, dtype: int64


,eval_case_id,consensus_error_type,error_type_votes,error_type_all3_agree,consensus_severity,severity_votes,severity_all3_agree,consensus_verdict,verdict_votes,verdict_all3_agree,...,agent3_model,agent3_error_type,agent3_severity,agent3_verdict,agent3_reason,row_id,scenario_slug,scenario_label,source_text,candidate_text
0,EVAL001,meaning_shift,3,True,major,3,True,perlu_ditinjau,3,True,...,Claude Sonnet 4.6,meaning_shift,major,perlu_ditinjau,Kata 'malillin' kemungkinan berarti 'memandika...,C33_T11_A6,fsp_locked,Fragment-Shot Prompting,Matumbara joq tedong malillin?,Bagaimana cara memasak kerbau?
1,EVAL002,hallucination,2,False,major,2,False,perlu_ditinjau,3,True,...,Claude Sonnet 4.6,meaning_shift,major,perlu_ditinjau,Frasa 'lako bokoqna' yang berarti 'ke arah bel...,C33_T14_B7,lora_locked,Fine-Tuning LoRA,Teqda. Ia to tanduk tedong malillin tuo lako b...,Tidak. Tanduk kerbau yang sudah besar akan kel...
2,EVAL003,untranslated,3,True,critical,2,False,perlu_ditinjau,3,True,...,Claude Sonnet 4.6,untranslated,critical,perlu_ditinjau,Kandidat terjemahan identik dengan source Baha...,C09_T11_A6,zero_shot,Zero-Shot,Pirang taunmira to muane?,Pirang taunmira to muane?
3,EVAL004,__no_majority__,1,False,__no_majority__,1,False,layak,2,False,...,Claude Sonnet 4.6,meaning_shift,major,perlu_ditinjau,Frasa 'teqda kitaq kuissenni' lebih tepat dite...,C03_T13_A7,fsp_locked,Fragment-Shot Prompting,"Teqda kitaq kuissenni, mbai den jao Juppandang.","Saya tidak tahu, mungkin ada di situ Juppandang."
4,EVAL005,hallucination,3,True,critical,3,True,perlu_ditinjau,3,True,...,Claude Sonnet 4.6,hallucination,critical,perlu_ditinjau,Source 'Umbori joq too?' tampaknya merupakan k...,C09_T13_A7,fsp_locked,Fragment-Shot Prompting,Umbori joq too?,"Saya tidak tahu, mungkin disekitar sini saja d..."


## 4. Pairwise agreement + unanimous rate

In [4]:
models = sorted(
    judgments[
        "requested_model"
    ].unique()
)

agreement_rows = []

for model_a, model_b in combinations(
    models,
    2,
):
    a = (
        judgments[
            judgments[
                "requested_model"
            ]
            == model_a
        ]
        .sort_values(
            "eval_case_id"
        )
        .reset_index(drop=True)
    )

    b = (
        judgments[
            judgments[
                "requested_model"
            ]
            == model_b
        ]
        .sort_values(
            "eval_case_id"
        )
        .reset_index(drop=True)
    )

    if (
        a["eval_case_id"].tolist()
        != b["eval_case_id"].tolist()
    ):
        raise RuntimeError(
            "Pairwise alignment gagal."
        )

    row = {
        "model_a": model_a,
        "model_b": model_b,
    }

    for attr in ATTRIBUTES:
        col = PRED_COLUMNS[attr]

        row[
            f"{attr}_agreement"
        ] = float(
            np.mean(
                a[col].values
                == b[col].values
            )
        )

    row[
        "joint_tuple_agreement"
    ] = float(
        np.mean(
            (
                a[
                    "pred_error_type"
                ].values
                ==
                b[
                    "pred_error_type"
                ].values
            )
            &
            (
                a[
                    "pred_severity"
                ].values
                ==
                b[
                    "pred_severity"
                ].values
            )
            &
            (
                a[
                    "pred_verdict"
                ].values
                ==
                b[
                    "pred_verdict"
                ].values
            )
        )
    )

    agreement_rows.append(row)

pairwise_agreement = pd.DataFrame(
    agreement_rows
)

all3_summary = {
    "error_type_all3_agree":
        float(
            case_level[
                "error_type_all3_agree"
            ].mean()
        ),
    "severity_all3_agree":
        float(
            case_level[
                "severity_all3_agree"
            ].mean()
        ),
    "verdict_all3_agree":
        float(
            case_level[
                "verdict_all3_agree"
            ].mean()
        ),
    "manual_review_rate":
        float(
            case_level[
                "needs_manual_review"
            ].mean()
        ),
}

display(pairwise_agreement)
print(all3_summary)

,model_a,model_b,error_type_agreement,severity_agreement,verdict_agreement,joint_tuple_agreement
0,anthropic/claude-sonnet-4.6,google/gemini-3.1-pro-preview,0.597484,0.553459,0.867925,0.503145
1,anthropic/claude-sonnet-4.6,openai/gpt-5.2,0.628931,0.591195,0.861635,0.446541
2,google/gemini-3.1-pro-preview,openai/gpt-5.2,0.610063,0.421384,0.880503,0.371069


{'error_type_all3_agree': 0.4591194968553459, 'severity_all3_agree': 0.31446540880503143, 'verdict_all3_agree': 0.8050314465408805, 'manual_review_rate': 0.1069182389937107}


## 5. Error/severity/verdict profile per skenario

In [5]:
def distribution_table(column):
    table = (
        case_level
        .groupby(
            [
                "scenario_slug",
                "scenario_label",
                column,
            ],
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
    )

    totals = (
        table
        .groupby(
            "scenario_slug"
        )["count"]
        .transform("sum")
    )

    table["proportion"] = (
        table["count"]
        / totals
    )

    return table

error_profile = distribution_table(
    "consensus_error_type"
)
severity_profile = distribution_table(
    "consensus_severity"
)
verdict_profile = distribution_table(
    "consensus_verdict"
)

scenario_summary = (
    case_level
    .groupby(
        [
            "scenario_slug",
            "scenario_label",
        ],
        as_index=False,
    )
    .agg(
        cases=(
            "eval_case_id",
            "count",
        ),
        error_type_all3_agreement=(
            "error_type_all3_agree",
            "mean",
        ),
        severity_all3_agreement=(
            "severity_all3_agree",
            "mean",
        ),
        verdict_all3_agreement=(
            "verdict_all3_agree",
            "mean",
        ),
        logical_consensus_rate=(
            "consensus_logical_valid",
            "mean",
        ),
        manual_review_rate=(
            "needs_manual_review",
            "mean",
        ),
    )
)

correct_rate = (
    case_level.assign(
        is_correct=
            case_level[
                "consensus_error_type"
            ].eq("correct")
    )
    .groupby(
        "scenario_slug"
    )["is_correct"]
    .mean()
)

layak_rate = (
    case_level.assign(
        is_layak=
            case_level[
                "consensus_verdict"
            ].eq("layak")
    )
    .groupby(
        "scenario_slug"
    )["is_layak"]
    .mean()
)

major_critical_rate = (
    case_level.assign(
        is_major_critical=
            case_level[
                "consensus_severity"
            ].isin(
                ["major", "critical"]
            )
    )
    .groupby(
        "scenario_slug"
    )["is_major_critical"]
    .mean()
)

scenario_summary["correct_rate"] = (
    scenario_summary[
        "scenario_slug"
    ].map(correct_rate)
)
scenario_summary["layak_rate"] = (
    scenario_summary[
        "scenario_slug"
    ].map(layak_rate)
)
scenario_summary[
    "major_critical_rate"
] = (
    scenario_summary[
        "scenario_slug"
    ].map(
        major_critical_rate
    )
)

display(scenario_summary)
display(error_profile)

,scenario_slug,scenario_label,cases,error_type_all3_agreement,severity_all3_agreement,verdict_all3_agreement,logical_consensus_rate,manual_review_rate,correct_rate,layak_rate,major_critical_rate
0,fsp_locked,Fragment-Shot Prompting,53,0.415094,0.358491,0.773585,0.867925,0.132075,0.113208,0.132075,0.811321
1,lora_locked,Fine-Tuning LoRA,53,0.283019,0.283019,0.716981,0.867925,0.132075,0.132075,0.132075,0.773585
2,zero_shot,Zero-Shot,53,0.679245,0.301887,0.924528,0.943396,0.056604,0.018868,0.018868,0.943396


,scenario_slug,scenario_label,consensus_error_type,count,proportion
0,fsp_locked,Fragment-Shot Prompting,__no_majority__,5,0.094340
1,fsp_locked,Fragment-Shot Prompting,correct,6,0.113208
2,fsp_locked,Fragment-Shot Prompting,fluency_issue,1,0.018868
3,fsp_locked,Fragment-Shot Prompting,hallucination,11,0.207547
4,fsp_locked,Fragment-Shot Prompting,meaning_shift,23,0.433962
5,fsp_locked,Fragment-Shot Prompting,missing_information,3,0.056604
6,fsp_locked,Fragment-Shot Prompting,source_leakage,3,0.056604
7,fsp_locked,Fragment-Shot Prompting,untranslated,1,0.018868
8,lora_locked,Fine-Tuning LoRA,__no_majority__,5,0.094340
9,lora_locked,Fine-Tuning LoRA,correct,7,0.132075


## 6. Merge otomatis 06D setelah evaluator selesai

In [6]:
FINAL_COMPARISON_PATH = (
    PROJECT_ROOT
    / "Hasil"
    / "final_comparison"
    / "final_comparison_metrics.csv"
)

combined_summary = (
    scenario_summary.copy()
)

if FINAL_COMPARISON_PATH.exists():
    automatic = pd.read_csv(
        FINAL_COMPARISON_PATH,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    required = {
        "scenario",
        "corpus_chrfpp",
        "corpus_bleu_13a",
        "mean_s3e",
    }

    if required.issubset(
        automatic.columns
    ):
        auto_small = automatic[
            [
                "scenario",
                "corpus_chrfpp",
                "corpus_bleu_13a",
                "mean_s3e",
            ]
        ].rename(
            columns={
                "scenario":
                    "scenario_slug",
            }
        )

        combined_summary = (
            combined_summary
            .merge(
                auto_small,
                on="scenario_slug",
                how="left",
                validate="one_to_one",
            )
        )

display(combined_summary)

,scenario_slug,scenario_label,cases,error_type_all3_agreement,severity_all3_agreement,verdict_all3_agreement,logical_consensus_rate,manual_review_rate,correct_rate,layak_rate,major_critical_rate,corpus_chrfpp,corpus_bleu_13a,mean_s3e
0,fsp_locked,Fragment-Shot Prompting,53,0.415094,0.358491,0.773585,0.867925,0.132075,0.113208,0.132075,0.811321,31.839264,10.565661,0.691243
1,lora_locked,Fine-Tuning LoRA,53,0.283019,0.283019,0.716981,0.867925,0.132075,0.132075,0.132075,0.773585,33.064789,9.191746,0.540777
2,zero_shot,Zero-Shot,53,0.679245,0.301887,0.924528,0.943396,0.056604,0.018868,0.018868,0.943396,18.976476,1.295909,0.741564


## 7. Save artifacts

In [7]:
CASE_PATH = (
    OUTPUT_DIR
    / "llm_evaluator_consensus_159.csv"
)
ERROR_PROFILE_PATH = (
    OUTPUT_DIR
    / "error_type_profile_by_scenario.csv"
)
SEVERITY_PROFILE_PATH = (
    OUTPUT_DIR
    / "severity_profile_by_scenario.csv"
)
VERDICT_PROFILE_PATH = (
    OUTPUT_DIR
    / "verdict_profile_by_scenario.csv"
)
AGREEMENT_PATH = (
    OUTPUT_DIR
    / "pairwise_evaluator_agreement.csv"
)
SCENARIO_SUMMARY_PATH = (
    OUTPUT_DIR
    / "scenario_evaluator_summary.csv"
)
COMBINED_PATH = (
    OUTPUT_DIR
    / "final_metrics_plus_llm_evaluator_summary.csv"
)
SUMMARY_JSON_PATH = (
    OUTPUT_DIR
    / "llm_evaluator_consensus_summary.json"
)

case_level.to_csv(
    CASE_PATH,
    index=False,
    encoding="utf-8-sig",
)
error_profile.to_csv(
    ERROR_PROFILE_PATH,
    index=False,
    encoding="utf-8-sig",
)
severity_profile.to_csv(
    SEVERITY_PROFILE_PATH,
    index=False,
    encoding="utf-8-sig",
)
verdict_profile.to_csv(
    VERDICT_PROFILE_PATH,
    index=False,
    encoding="utf-8-sig",
)
pairwise_agreement.to_csv(
    AGREEMENT_PATH,
    index=False,
    encoding="utf-8-sig",
)
scenario_summary.to_csv(
    SCENARIO_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)
combined_summary.to_csv(
    COMBINED_PATH,
    index=False,
    encoding="utf-8-sig",
)

summary_payload = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "cases": 159,
    "agents": 3,
    "consensus_method":
        "attribute-wise majority; 2/3 or 3/3",
    "no_majority_token":
        "__no_majority__",
    "unresolved_rule":
        (
            "no majority or logically inconsistent "
            "consensus -> needs_manual_review=True"
        ),
    "all3_agreement":
        all3_summary,
    "scenario_summary":
        scenario_summary.to_dict(
            orient="records"
        ),
    "source_judgments":
        str(FINAL_477_PATH),
    "outputs": {
        "case_level":
            str(CASE_PATH),
        "error_profile":
            str(ERROR_PROFILE_PATH),
        "severity_profile":
            str(SEVERITY_PROFILE_PATH),
        "verdict_profile":
            str(VERDICT_PROFILE_PATH),
        "agreement":
            str(AGREEMENT_PATH),
        "scenario_summary":
            str(SCENARIO_SUMMARY_PATH),
        "combined_summary":
            str(COMBINED_PATH),
    },
}

SUMMARY_JSON_PATH.write_text(
    json.dumps(
        summary_payload,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("CASE     :", CASE_PATH)
print("SUMMARY  :", SCENARIO_SUMMARY_PATH)
print("COMBINED :", COMBINED_PATH)
print("JSON     :", SUMMARY_JSON_PATH)

CASE     : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/consensus/llm_evaluator_consensus_159.csv
SUMMARY  : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/consensus/scenario_evaluator_summary.csv
COMBINED : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/consensus/final_metrics_plus_llm_evaluator_summary.csv
JSON     : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/consensus/llm_evaluator_consensus_summary.json


## 8. Final audit

In [8]:
checks = {
    "case_level_159":
        len(case_level) == 159,
    "three_scenarios":
        case_level[
            "scenario_slug"
        ].nunique() == 3,
    "53_each":
        bool(
            (
                case_level
                .groupby(
                    "scenario_slug"
                )
                .size()
                == 53
            ).all()
        ),
    "three_agents":
        judgments[
            "requested_model"
        ].nunique() == 3,
    "477_judgments":
        len(judgments) == 477,
    "case_file_exists":
        CASE_PATH.exists(),
    "summary_exists":
        SCENARIO_SUMMARY_PATH.exists(),
    "combined_exists":
        COMBINED_PATH.exists(),
}

failed = [
    k for k, v in checks.items()
    if not bool(v)
]

display(
    pd.DataFrame(
        [
            {"check": k, "ok": bool(v)}
            for k, v in checks.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL CONSENSUS AUDIT GAGAL: {failed}"
    )

print("\n" + "=" * 78)
print("3-AGENT CONSENSUS + ERROR ANALYSIS: COMPLETE")
print("=" * 78)
display(combined_summary)

unresolved = int(
    case_level[
        "needs_manual_review"
    ].sum()
)

print(
    "\nCases needing manual review:",
    unresolved,
    "/ 159",
)

,check,ok
0,case_level_159,True
1,three_scenarios,True
2,53_each,True
3,three_agents,True
4,477_judgments,True
5,case_file_exists,True
6,summary_exists,True
7,combined_exists,True



3-AGENT CONSENSUS + ERROR ANALYSIS: COMPLETE


,scenario_slug,scenario_label,cases,error_type_all3_agreement,severity_all3_agreement,verdict_all3_agreement,logical_consensus_rate,manual_review_rate,correct_rate,layak_rate,major_critical_rate,corpus_chrfpp,corpus_bleu_13a,mean_s3e
0,fsp_locked,Fragment-Shot Prompting,53,0.415094,0.358491,0.773585,0.867925,0.132075,0.113208,0.132075,0.811321,31.839264,10.565661,0.691243
1,lora_locked,Fine-Tuning LoRA,53,0.283019,0.283019,0.716981,0.867925,0.132075,0.132075,0.132075,0.773585,33.064789,9.191746,0.540777
2,zero_shot,Zero-Shot,53,0.679245,0.301887,0.924528,0.943396,0.056604,0.018868,0.018868,0.943396,18.976476,1.295909,0.741564



Cases needing manual review: 17 / 159
